# Inspect Article-Level Content Classification

Reads the final zero-shot content-classification outputs, checks distributions and missingness, and previews the merged article-level measurement dataset.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_PATH = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in [NOTEBOOK_PATH, *NOTEBOOK_PATH.parents] if (path / 'config.py').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 180)

CONTENT_DIR = Path('/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/content_classification/final_gpt51')

VICTIM_PATH = CONTENT_DIR / 'victim_visibility_labels.csv.gz'
FRAME_PATH = CONTENT_DIR / 'corruption_frame_labels.csv.gz'
ABROAD_PATH = CONTENT_DIR / 'abroad_case_labels.csv.gz'
ACCUSED_PATH = CONTENT_DIR / 'accused_actor_labels.csv.gz'
MERGED_PATH = CONTENT_DIR / 'political_corruption_content_categories_final.csv.gz'

CONTENT_DIR

In [ ]:
paths = {
    'victim': VICTIM_PATH,
    'frame': FRAME_PATH,
    'abroad': ABROAD_PATH,
    'accused': ACCUSED_PATH,
}

tables = {}
for name, path in paths.items():
    if path.exists():
        tables[name] = pd.read_csv(path)
        print(f'{name}: {len(tables[name]):,} rows from {path}')
    else:
        print(f'Missing {name}: {path}')

In [ ]:
for name, data in tables.items():
    display(Markdown(f'## {name}'))
    print(f'Rows: {len(data):,}')
    if 'llm_error' in data.columns:
        print('Errors:', data['llm_error'].fillna('').astype(str).str.strip().ne('').sum())
    label_cols = [
        col for col in data.columns
        if col.endswith('_visibility') or col.endswith('_visible') or col in {'corruption_frame', 'case_location', 'abroad_case'}
    ]
    for col in label_cols:
        display(data[col].value_counts(dropna=False).to_frame('n'))

In [ ]:
if MERGED_PATH.exists():
    merged = pd.read_csv(MERGED_PATH)
    print(f'Merged rows: {len(merged):,}')
    display(merged.head())
    derived_cols = [
        'victim_visible_binary',
        'concrete_victim_visible',
        'institutional_societal_victim_visible',
        'abroad_case_binary',
        'accused_actor_visible_binary',
        'frame_individualized',
        'frame_systemic',
        'perceived_corruption_lag1',
    ]
    display(merged[[col for col in derived_cols if col in merged.columns]].describe(include='all'))
else:
    print(f'Merged file not found yet: {MERGED_PATH}')